# Imports

In [2]:
import re

import pandas as pd

pd.options.display.max_columns = 50

# Download data

## Human (GENCODE, high-quality protein data)

In [1]:
! aws s3 cp s3://seanome-kmerseek/applications/botryllus/gencode/v38/gencode.v38.basic.annotation.protein.fa \
    ~/s3/seanome-kmerseek/applications/botryllus/data/gencode/v38/gencode.v38.basic.annotation.protein.fa

download: s3://seanome-kmerseek/applications/botryllus/gencode/v38/gencode.v38.basic.annotation.protein.fa to ../../s3/seanome-kmerseek/applications/botryllus/data/gencode/v38/gencode.v38.basic.annotation.protein.fa


In [10]:
human_fasta = "/home/ec2-user/s3/seanome-kmerseek/applications/botryllus/data/gencode/v38/gencode.v38.basic.annotation.protein.fa"

## *Botryllus schlosseri* (early marine organism) gene: *Botryllus Histocompatibility Factor* (BHF)

In [2]:
! aws s3 cp s3://seanome-kmerseek/applications/botryllus/botryllus-proteins/bhf.fa  \
    ~/s3/seanome-kmerseek/applications/botryllus/botryllus-proteins/bhf.fa

download: s3://seanome-kmerseek/applications/botryllus/botryllus-proteins/bhf.fa to ../../s3/seanome-kmerseek/applications/botryllus/botryllus-proteins/bhf.fa


In [18]:
bhf_fasta = "/home/ec2-user/s3/seanome-kmerseek/applications/botryllus/botryllus-proteins/bhf.fa"

# Make files to input to manysketch

## Human

In [3]:
%%file $human_fasta.manysketch.csv
name,genome_filename,protein_filename
human,,$human_fasta

Overwriting /home/ec2-user/s3/seanome-kmerseek/applications/botryllus/data/gencode/v38/gencode.v38.basic.annotation.protein.fa.manysketch.csv


## BHF

In [4]:
%%file $bhf_fasta.manysketch.csv
name,genome_filename,protein_filename
BHF,,$bhf_fasta

Overwriting /home/ec2-user/s3/seanome-kmerseek/applications/botryllus/botryllus-proteins/bhf.fa.manysketch.csv


# Run Manysketch

In [14]:
! sourmash scripts manysketch  --help


== This is sourmash version 4.8.14. ==
== Please cite Irber et. al (2024), doi:10.21105/joss.06830. ==

usage:  manysketch [-h] [-q] [-d] -o OUTPUT [-p PARAM_STRING] [-c CORES] [-s]
                   [-f]
                   fromfile_csv

massively parallel sketching

positional arguments:
  fromfile_csv          a csv file containing paths to FASTA files. Columns
                        must be: 'name,genome_filename,protein_filename' or
                        'name,read1,read2'

options:
  -h, --help            show this help message and exit
  -q, --quiet           suppress non-error output
  -d, --debug           provide debugging output
  -o OUTPUT, --output OUTPUT
                        output zip file for the signatures
  -p PARAM_STRING, --param-string PARAM_STRING
                        parameter string for sketching (default:
                        k=31,scaled=1000)
  -c CORES, --cores CORES
                        number of cores to use (default is all available)
  -s, 

## Set parameters to share

In [15]:
parameters = "--singleton -p hp,k=24,scaled=5,abund"

## BHF

In [59]:
%%time

bhf_sig = f"{bhf_fasta}.hp.k24.scale5.zip"
output_zip = bhf_sig
input_csv = f"{bhf_fasta}.manysketch.csv"

! sourmash scripts manysketch $parameters \
    -o $output_zip \
    $input_csv


== This is sourmash version 4.8.14. ==
== Please cite Irber et. al (2024), doi:10.21105/joss.06830. ==

=> sourmash_plugin_branchwater 0.9.14.dev0; cite Irber et al., doi: 10.1101/2022.11.02.514947

params: ['hp,k=24,scaled=5,abund']
sketching all files in '/home/ec2-user/s3/seanome-kmerseek/applications/botryllus/botryllus-proteins/bhf.fa.manysketch.csv' using 8 threads
Loaded 1 rows in total (0 genome and 1 protein files)
Building 1 sketch types:
    hp,k=24,scaled=5,num=0,abund=true
Starting file 1/1 (100%)
Writing manifest
DONE. Processed 1 fasta files
...manysketch is done! results in '/home/ec2-user/s3/seanome-kmerseek/applications/botryllus/botryllus-proteins/bhf.fa.hp.k24.scale5.zip'
CPU times: user 10.3 ms, sys: 762 μs, total: 11.1 ms
Wall time: 582 ms


## Human: Singleton

Human is the reference database, but would be precomputed in a "real" query. Mostly this is to show that the "Training" step takes fewer resources than LLMs :)

In [57]:
%%time

human_singleton_sig = f"{human_fasta}.hp.k24.scale5.singleton.zip"
output_zip = human_singleton_sig
input_csv = f"{human_fasta}.manysketch.csv"

! sourmash scripts manysketch $parameters \
    -o $output_zip \
    $input_csv


== This is sourmash version 4.8.14. ==
== Please cite Irber et. al (2024), doi:10.21105/joss.06830. ==

=> sourmash_plugin_branchwater 0.9.14.dev0; cite Irber et al., doi: 10.1101/2022.11.02.514947

params: ['hp,k=24,scaled=5,abund']
sketching all files in '/home/ec2-user/s3/seanome-kmerseek/applications/botryllus/data/gencode/v38/gencode.v38.basic.annotation.protein.fa.manysketch.csv' using 8 threads
Loaded 1 rows in total (0 genome and 1 protein files)
Building 1 sketch types:
    hp,k=24,scaled=5,num=0,abund=true
Starting file 1/1 (100%)
Writing manifest
DONE. Processed 1 fasta files
...manysketch is done! results in '/home/ec2-user/s3/seanome-kmerseek/applications/botryllus/data/gencode/v38/gencode.v38.basic.annotation.protein.fa.hp.k24.scale5.singleton.zip'
CPU times: user 719 ms, sys: 165 ms, total: 883 ms
Wall time: 1min 32s


## Human: Aggregated

This signature is to be used to compute the background probability of each k-mer

In [77]:
%%time

human_aggregated_sig = f"{human_fasta}.hp.k24.scale5.aggregated.zip"
output_zip = human_aggregated_sig
input_csv = f"{human_fasta}.manysketch.csv"

! sourmash scripts manysketch -p hp,k=24,scaled=5,abund \
    -o $output_zip \
    $input_csv


== This is sourmash version 4.8.14. ==
== Please cite Irber et. al (2024), doi:10.21105/joss.06830. ==

=> sourmash_plugin_branchwater 0.9.14.dev0; cite Irber et al., doi: 10.1101/2022.11.02.514947

params: ['hp,k=24,scaled=5,abund']
sketching all files in '/home/ec2-user/s3/seanome-kmerseek/applications/botryllus/data/gencode/v38/gencode.v38.basic.annotation.protein.fa.manysketch.csv' using 8 threads
Loaded 1 rows in total (0 genome and 1 protein files)
Building 1 sketch types:
    hp,k=24,scaled=5,num=0,abund=true
Starting file 1/1 (100%)
Writing manifest
DONE. Processed 1 fasta files
...manysketch is done! results in '/home/ec2-user/s3/seanome-kmerseek/applications/botryllus/data/gencode/v38/gencode.v38.basic.annotation.protein.fa.hp.k24.scale5.aggregated.zip'
CPU times: user 415 ms, sys: 141 ms, total: 556 ms
Wall time: 57.1 s


# Run `fastgather`

# Run `multisearch`

In [4]:
! sourmash scripts multisearch --help


== This is sourmash version 4.8.14. ==
== Please cite Irber et. al (2024), doi:10.21105/joss.06830. ==

usage:  multisearch [-h] [-q] [-d] -o OUTPUT [-t THRESHOLD] [-k KSIZE]
                    [-s SCALED]
                    [-m {DNA,protein,dayhoff,hp,skipm1n3,skipm2n3}] [-c CORES]
                    [-a] [-p] [-A]
                    query_paths against_paths

massively parallel in-memory sketch search

positional arguments:
  query_paths           input file of sketches
  against_paths         input file of sketches

options:
  -h, --help            show this help message and exit
  -q, --quiet           suppress non-error output
  -d, --debug           provide debugging output
  -o OUTPUT, --output OUTPUT
                        CSV output file for matches
  -t THRESHOLD, --threshold THRESHOLD
                        containment threshold for reporting matches (default:
                        0.01)
  -k KSIZE, --ksize KSIZE
                        k-mer size at which to select

In [ ]:
! sourmash scripts multisearch \
    --threshold 0 \
    --ksize 24 \
    --moltype hp \
    --prob-significant-overlap \
    $

# Get kmers for each hashval

In [66]:
! python sig2kmer.py --help

usage: sig2kmer.py [-h] [--output-sequences OUTPUT_SEQUENCES]
                   [--output-kmers OUTPUT_KMERS] [--input-is-protein] [-k K]
                   [--protein] [--no-protein] [--dayhoff] [--no-dayhoff]
                   [--hp] [--no-hp] [--skipm1n3] [--no-skipm1n3] [--skipm2n3]
                   [--no-skipm2n3] [--dna] [--no-dna]
                   query seqfiles [seqfiles ...]

positional arguments:
  query
  seqfiles

options:
  -h, --help            show this help message and exit
  --output-sequences OUTPUT_SEQUENCES
                        save matching sequences to this file.
  --output-kmers OUTPUT_KMERS
                        save matching kmers to this file.
  --input-is-protein    Consume protein sequences - no translation needed.
  -k K, --ksize K       k-mer size to select; no default.
  --protein             choose a protein signature; by default, a nucleotide
                        signature is used
  --no-protein          do not choose a protein signature
 

## Get BHF kmers

In [83]:
%%time

bhf_kmers_csv = f"{bhf_sig}.kmers.csv"
! python sig2kmer.py \
    --output-kmers $bhf_kmers_csv \
    --no-dna \
    --input-is-protein \
    --hp \
    -k 24 \
    $bhf_sig \
    $bhf_fasta

CPU times: user 13.3 ms, sys: 0 ns, total: 13.3 ms
Wall time: 885 ms


### Read BHF Kmer csv

In [84]:
bhf_kmers = pd.read_csv(bhf_kmers_csv)
print(bhf_kmers.shape)
bhf_kmers.head()

(40, 6)


,kmer_in_sequence,kmer_in_alphabet,hashval,start,read_name,filename
0,VHDTEQLLAQGHHEEETECGKYGK,hppppphhhphpppppppphphhp,1036020595944595459,1,BHF,/home/ec2-user/s3/seanome-kmerseek/application...
1,TEQLLAQGHHEEETECGKYGKLPE,ppphhhphpppppppphphhphhp,2993707203445337902,4,BHF,/home/ec2-user/s3/seanome-kmerseek/application...
2,EQLLAQGHHEEETECGKYGKLPEK,pphhhphpppppppphphhphhpp,973051056292048589,5,BHF,/home/ec2-user/s3/seanome-kmerseek/application...
3,ETECGKYGKLPEKGSECKKHGIFC,pppphphhphhpphpppppphhhp,385192726330471768,16,BHF,/home/ec2-user/s3/seanome-kmerseek/application...
4,KYGKLPEKGSECKKHGIFCRILTA,phhphhpphpppppphhhpphhph,1194029314525711201,21,BHF,/home/ec2-user/s3/seanome-kmerseek/application...


## Get human kmers

In [ ]:
human_aggregated_kmers_csv = f"{human_aggregated_sig}.kmers.csv"

In [ ]:
%%time

! python sig2kmer.py \
    --output-kmers $human_aggregated_kmers_csv \
    --no-dna \
    --input-is-protein \
    --hp \
    -k 24 \
    $human_aggregated_sig \
    $human_fasta

CPU times: user 1.84 s, sys: 387 ms, total: 2.23 sers in	61358 seqs from	/home/ec2-user/s3/seanome-kmerseek/applications/botryllus/data/gencode/v38/gencode.v38.basic.annotation.protein.fa
Wall time: 3min 46s


### Read Human Kmer csv

In [3]:
human_aggregated_kmers_csv = '/home/ec2-user/s3/seanome-kmerseek/applications/botryllus/data/gencode/v38/gencode.v38.basic.annotation.protein.fa

human_aggregated_kmers = pd.read_csv(human_aggregated_kmers_csv)
print(human_aggregated_kmers.shape)
human_aggregated_kmers.head()

NameError: name 'human_aggregated_kmers_csv' is not defined